# Contextual median fraud prediction
Prediction-only frozen exploratory model.

In [ ]:
import os,pickle
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.special import logsumexp
with open('model.pkl','rb') as f:bundle=pickle.load(f)
TARGETS=['transaction_amount','transactions_last_1h']
LOG=['transaction_amount','spend_last_24h','account_age','transactions_last_24h','transactions_last_1h']
CAT=['merchant_category','country','transaction_channel']
def target_values(X,target):
    a=pd.to_numeric(X[target],errors='coerce').to_numpy(float);a=np.where(np.isfinite(a)&(a>=0),a,np.nan);return np.log1p(a)


def features(X,target):
    F=X[CAT].copy()
    for col in LOG:
        if col!=target:F['log_'+col]=target_values(X,col)
    F['new_device']=pd.to_numeric(X.new_device,errors='coerce').replace([np.inf,-np.inf],np.nan)
    hr=pd.to_numeric(X.transaction_hour,errors='coerce').to_numpy(float);hr=np.where(np.isfinite(hr)&(hr>=0)&(hr<24),hr,np.nan)
    F['hour_sin']=np.sin(2*np.pi*hr/24);F['hour_cos']=np.cos(2*np.pi*hr/24)
    return F


def encode(frame,state):
    out=np.full((len(frame),len(state)),-1,dtype=np.int64)
    for i,item in enumerate(state):
        s=frame[item['name']];missing=s.isna().to_numpy();kind=item['kind']
        if kind=='category':
            mapping={v:j for j,v in enumerate(item['levels'])}
            out[:,i]=s.map(lambda v:mapping.get(str(v),-1) if not pd.isna(v) else -1).to_numpy(int)
        else:
            a=pd.to_numeric(s,errors='coerce').to_numpy(float);ok=np.isfinite(a)
            if kind=='numeric':out[ok,i]=np.searchsorted(item['cuts'],a[ok],side='right')
            elif kind=='hour':
                ok &= (a>=0)&(a<24);out[ok,i]=np.floor(((a[ok]+3)%24)/6).astype(int)
            else:
                ok &= (a==0)|(a==1);out[ok,i]=a[ok].astype(int)
        if item['missing_mode']=='state':out[missing,i]=item['value_states']
    return out


def postorder(parent):
    children=[[] for _ in parent]
    for i,p in enumerate(parent):
        if p>=0:children[p].append(i)
    order=[]
    def visit(i):
        for j in children[i]:visit(j)
        order.append(i)
    for i,p in enumerate(parent):
        if p<0:visit(i)
    assert len(order)==len(parent) and len(set(order))==len(parent)
    return order,children


def log_evidence(model,x):
    """Exact evidence likelihood for arbitrary observed subsets and either class."""
    x=np.asarray(x,dtype=np.int64);ks=model['ks'];parent=model['parent']
    assert x.ndim==2 and x.shape[1]==len(ks)
    assert all(np.all((x[:,i]>=-1)&(x[:,i]<k)) for i,k in enumerate(ks))
    order,children=postorder(parent);answer=np.zeros((len(x),2))
    for c in range(2):
        messages={}
        for i in order:
            local=np.zeros((len(x),ks[i]));known=x[:,i]>=0
            local[known,:]=-np.inf;local[np.flatnonzero(known),x[known,i]]=0.
            for child in children[i]:local+=messages[child]
            t=np.log(model['probability'][i][c])
            if parent[i]<0:answer[:,c]+=logsumexp(local+t,axis=1)
            else:messages[i]=logsumexp(local[:,None,:]+t[None,:,:],axis=2)
    return answer


def predict_encoded(model,x):
    lp=log_evidence(model,x)+np.log(model['prior'])
    return np.exp(lp-logsumexp(lp,axis=1,keepdims=True))


def predict_proba(model,frame):
    return predict_encoded(model,encode(frame,model['encoder']))

def predict_fraud(frame):
    x=frame[bundle['feature_columns']];normal=[]
    for state in bundle['normal_models']:
        a=state['preprocess'].transform(features(x,state['target']))
        normal.append(state['models'][0].predict(a))
    q50=np.column_stack(normal)
    targets=np.column_stack([target_values(x,name) for name in TARGETS])
    a=bundle['preprocess'].transform(x);b=bundle['basis']
    residual=b['imputer'].transform(targets-q50)-np.column_stack([np.ones(len(x)),a])@b['projection']
    residual=b['scaler'].transform(residual[:,b['keep']])
    lr=bundle['classifier'].predict_proba(np.column_stack([a,.2*residual]))[:,1]
    cp=bundle['cat_pipeline'].predict_proba(x)[:,1];tp=predict_proba(bundle['tan'],x)[:,1]
    p=.63*lr+.27*cp+.1*tp
    if len(p)!=len(x) or not np.isfinite(p).all() or not ((p>=0)&(p<=1)).all():raise ValueError('Invalid fraud probabilities')
    return p
frame=pd.read_csv(Path(os.environ.get('DATATHON_INPUT_PATH','test.csv')))
scores=predict_fraud(frame)
pd.DataFrame({'prediction':scores}).to_csv(Path(os.environ.get('DATATHON_OUTPUT_PATH','predictions.csv')),index=False,float_format='%.17g')
print(f'Wrote {len(scores)} fraud probabilities')
